In [1]:
# Conexão e Dataframe 10.10.1.87
# Query Matriculas

import pyodbc
import pandas as pd

%run Caminhos.ipynb

dados_conexao_seduc

try:
    # Abrir conexão
    conexao = pyodbc.connect(dados_conexao_seduc)
    print("Conexão bem sucedida")

    # Executa uma consulta
    cursor = conexao.cursor()

    # Executa o Select
    query = """
    SELECT
        m.idMatricula, 
        m.idAluno, 
        m.idTurma,
        t.nome as 'Nome da turma',
        est.idEstado as 'Id Estado',
        est.nome as 'Estado',
        mu.territorioDesenvolvimento as 'Territorio Desenvolv', 
        gr.nome as 'Gerencia Regional', 
        CAST(mu.idMunicipio as INT) as 'Id Municipio', 
        mu.nome as 'Municipio', 
        es.inep,
        es.idEscola as 'Id Escola',
        es.nome as 'Escola',
        es.ativa,
        es.localizacao,
        pl.idPeriodo, 
        CAST(pl.anoLetivo AS INT) as 'Ano', 
        m.matriculaAtiva, 
        m.situacaoMatricula, 
        m.situacaoEnturmacao,
        e.idEtapa,
        e.nome as 'NomeEtapa',
        e.modalidadeEnsino, 
        e.modalidadeEnsinoISeduc,
        e.grupo,
        e.nivelEnsino, 
        e.ordem,
        e.agregada as 'EtapaAgregada', 
        e.curso as 'NomeCurso',
        CASE 
            WHEN e.tempoIntegral = 1 THEN 'Tempo Integral' 
            WHEN e.tempoIntegral = 0 THEN 'Parcial' 
            ELSE 'Null' 
        END AS 'TempoIntegral',
        CASE
            WHEN es.tempoIntegral = 1 AND e.tempoIntegral = 1 THEN 1
            ELSE 0
        END AS STATUS_INTEGRAL,
        es.tempoIntegral as 'tempoIntegralEscola'
    FROM matricula m 
        LEFT JOIN periodoLetivo pl ON pl.idPeriodo = m.idPeriodo AND pl.anoLetivo in (2024) 
        LEFT JOIN etapa e ON e.idEtapa = m.idEtapa 
        LEFT JOIN escola es ON es.idEscola =  m.idEscola 
        LEFT JOIN municipio mu ON mu.idMunicipio = es.idMunicipio 
        LEFT JOIN gerenciaRegional gr ON gr.idGerenciaRegional = es.idGerenciaRegional
        LEFT JOIN estado est ON est.idEstado = mu.idEstado
        LEFT JOIN turma t ON t.idTurma = m.idturma
        WHERE
            es.dependenciaAdministrativa = 'ESTADUAL'
        ORDER BY 
            est.idEstado,
            m.idMatricula, 
            mu.idMunicipio,
            mu.nome, 
            es.inep,
            es.idEscola, 
            es.nome, 
            es.ativa,
            es.localizacao,
            pl.idPeriodo, 
            pl.anoLetivo, 
            m.matriculaAtiva, 
            m.situacaoMatricula, 
            m.situacaoEnturmacao, 
            e.idEtapa, 
            e.nome, 
            e.modalidadeEnsino, 
            e.modalidadeEnsinoISeduc, 
            e.grupo,
            e.nivelEnsino, 
            e.agregada, 
            e.curso, 
            e.tempoIntegral
    """
    cursor.execute(query)

    # Obter os nomes das colunas
    columns = [desc[0] for desc in cursor.description]
    select = cursor.fetchall()

    # Desmebra a lista de Tuplas em uma tabela
    tabela_clientes = pd.DataFrame.from_records(select, columns=columns)
    
    # Adicionar uma nova coluna com base na condição 2025
    tabela_clientes['Matricula_2026'] = tabela_clientes.apply(
        lambda row: 1 if row['Ano'] == 2026 and row['situacaoMatricula'] == "Cursando" and row['matriculaAtiva'] == 1 else 0,
        axis=1
    )

    # Adicionar uma nova coluna com base na condição 2025
    tabela_clientes['Matricula_2025'] = tabela_clientes.apply(
        lambda row: 1 if row['Ano'] == 2025 and row['situacaoMatricula'] == "Cursando" and row['matriculaAtiva'] == 1 else 0,
        axis=1
    )

    # Adicionar uma nova coluna com base na condição 2024
    tabela_clientes['Matricula_2024'] = tabela_clientes.apply(
        lambda row: 1 if row['Ano'] == 2024 and row['situacaoMatricula'] == "Cursando" and row['matriculaAtiva'] == 1 else 0,
        axis=1
    )

    # Adicionar uma nova coluna com base na condição 2023
    tabela_clientes['Matricula_2023'] = tabela_clientes.apply(
        lambda row: 1 if row['Ano'] == 2023 and row['matriculaAtiva'] == 1 and row['situacaoMatricula'] in ("Aprovado", "Concluído", "Cursando", "Reprovado") else 0,
        axis=1
    )

    # Adicionar a etapa 3 - MTMatricula
    def calculate_MTMatricula(row):
        if row['Ano'] == 2026 and row['Matricula_2026'] == 1:
            return 1
        elif row['Ano'] == 2025 and row['Matricula_2025'] == 1:
            return 1
        elif row['Ano'] == 2024 and row['Matricula_2024'] == 1:
            return 1
        elif row['Ano'] == 2023 and row['Matricula_2023'] == 1:
            return 1
        else:
            return 0

    tabela_clientes['MTMatricula'] = tabela_clientes.apply(calculate_MTMatricula, axis=1) 

except pyodbc.OperationalError as erro:
    # Captura e trata erros operacionais específicos do pyodbc
    print(f"Erro operacional ao criar a conexão: {erro}")

except Exception as erro:
    # Captura e trata qualquer outro tipo de exceção
    print(f"Erro ao criar a conexão ou executar a consulta: {erro}")

finally:
    # Fecha a conexão com o banco de dados, se estiver aberta
    if cursor:
        cursor.close()
        conexao.close()

%run Caminhos.ipynb

# salvar os dados em um arquivo pickle
#%timeit tabela_clientes.to_pickle(f'{pasta_hierarquia}{caminho_tabela_clientes_pkl}')

# salvar os dados em um arquivo parquet
%timeit tabela_clientes.to_parquet(f'{pasta_hierarquia}{caminho_tabela_clientes_parquet}')

# salvar os dados em um arquivo parquet
%timeit tabela_clientes.to_csv(f'{pasta_hierarquia}{caminho_tabela_clientes_csv}')

# salvar os dados em um arquivo feather
#%timeit tabela_clientes.to_feather(f'{pasta_hierarquia}{caminho_tabela_clientes_feather}')

#!dir "C:/Users/VeryPC/OneDrive - verytecnologia.com.br/4.PIAUÍ - 1.SEDUC/2. Execução/2.5 Projetos/Escritório de Projetos/4. Painéis/Painel iSEDUC/Indicadores_Hierarquia/Query_indicadores/"

Erro ao conectar: ORA-12545: Connect failed because target host or object does not exist
Help: https://docs.oracle.com/error-help/db/ora-12545/
Conexão bem sucedida
Erro ao conectar: ORA-12545: Connect failed because target host or object does not exist
Help: https://docs.oracle.com/error-help/db/ora-12545/
4.51 s ± 542 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
18.8 s ± 4.51 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [4]:
# Conexão e Dataframe 10.10.1.58
# Query Especificação Indicadores

import pyodbc
import pandas as pd

%run Caminhos.ipynb

dados_conexao_brisk

try:
    # Abrir conexão
    conexao = pyodbc.connect(dados_conexao_brisk)
    print("Conexão bem sucedida")

    # Executa uma consulta
    cursor = conexao.cursor()

    # Executa o Select
    query = """
WITH CTE AS (
    SELECT 
        ri.Ano,
        ri.Mes,
        ri.CodigoIndicador,
        ri.CodigoUnidadeNegocio,
        ri.MetaMes,
        LAG(ri.MetaMes) OVER (PARTITION BY ri.CodigoIndicador, ri.Ano ORDER BY ri.Mes) AS MetaMes_Lag
    FROM 
        ResumoIndicador ri
),

Subconsulta AS (
    SELECT 
        rr.Ano,
        rr.Mes,
        rr.ResultadoMes,
        COALESCE(rr.MetaMes, LAG(rr.MetaMes) OVER (PARTITION BY rr.CodigoIndicador, rr.Ano ORDER BY rr.Mes)) AS MetaMes, -- Preenche os valores nulos com o trimestre
        CASE 
            WHEN rr.MetaMes IS NULL THEN rr.MetaMes
            WHEN ii.IndicaCriterio = 'S' THEN rr.MetaMes  
            ELSE SUM(rr.MetaMes) OVER (PARTITION BY rr.CodigoIndicador, rr.Ano ORDER BY rr.Mes) 
        END AS MetaAcumuladaAno,
        rr.CodigoIndicador,
        rr.CodigoUnidadeNegocio
    FROM 
        ResumoIndicador rr
    LEFT JOIN 
        Indicador ii ON ii.CodigoIndicador = rr.CodigoIndicador
    WHERE
        rr.CodigoUnidadeNegocio <> 447
)
       
SELECT 
un.CodigoUnidadeNegocio,
un.CodigoUnidadeNegocioSuperior,
un.SiglaUnidadeNegocio AS 'SiglaUN',
un.NomeUnidadeNegocio AS 'NomeUnidade',
un.CorUnidadeNegocio,
ri.CodigoIndicador AS 'Codigo Indicador',
i.NomeIndicador AS 'Indicador',
ri.Ano AS 'Ano_Meta',
ri.Mes AS 'Mes_Meta',
(SELECT TOP 1
	p.CodigoProjeto
	FROM LinkFormulario LF
	LEFT JOIN Formulario f on f.CodigoFormulario = LF.CodigoFormulario
	left join Formulario f2 on f2.CodigoFormulario = lf.CodigoSubFormulario
	left join CampoFormulario cf on cf.CodigoFormulario = f2.CodigoFormulario
	left join FormulariosInstanciasWorkflows fiw on fiw.CodigoFormulario = f.CodigoFormulario
	left join InstanciasWorkflows inw on inw.CodigoInstanciaWf = fiw.CodigoInstanciaWf
	left join FormularioProjeto fp on f.CodigoFormulario = fp.CodigoFormulario
	left join projeto p on fp.CodigoProject = p.CodigoProjeto
	LEFT JOIN Indicador ie ON cf.ValorCampoSomenteLeitura = CAST(ie.CodigoIndicador AS VARCHAR)

	where f.CodigoModeloFormulario in (4714,4713)
	and f2.CodigoModeloFormulario in (4714,4713)
    AND f.DataExclusao IS NULL
	AND f2.DataExclusao IS NULL
    AND p.DataExclusao IS NULL
	and inw.DataCancelamentoInstancia IS NULL
	and inw.DataTerminoInstancia IS NOT NULL
    AND ie.NomeIndicador IS NOT NULL
	AND ie.CodigoIndicador = ri.CodigoIndicador
	ORDER BY
	cf.ValorCampoSomenteLeitura,
	inw.DataTerminoInstancia ASC) as 'CodigoProjeto',

	(SELECT TOP 1
	P.NomeProjeto
	FROM LinkFormulario LF
	LEFT JOIN Formulario f on f.CodigoFormulario = LF.CodigoFormulario
	left join Formulario f2 on f2.CodigoFormulario = lf.CodigoSubFormulario
	left join CampoFormulario cf on cf.CodigoFormulario = f2.CodigoFormulario
	left join FormulariosInstanciasWorkflows fiw on fiw.CodigoFormulario = f.CodigoFormulario
	left join InstanciasWorkflows inw on inw.CodigoInstanciaWf = fiw.CodigoInstanciaWf
	left join FormularioProjeto fp on f.CodigoFormulario = fp.CodigoFormulario
	left join projeto p on fp.CodigoProject = p.CodigoProjeto
	LEFT JOIN Indicador ie ON cf.ValorCampoSomenteLeitura = CAST(ie.CodigoIndicador AS VARCHAR)
	where f.CodigoModeloFormulario in (4714,4713)
	and f2.CodigoModeloFormulario in (4714,4713)
    AND f.DataExclusao IS NULL
	AND f2.DataExclusao IS NULL
    AND p.DataExclusao IS NULL
	and inw.DataCancelamentoInstancia IS NULL
	and inw.DataTerminoInstancia IS NOT NULL
    AND ie.NomeIndicador IS NOT NULL
	AND ie.CodigoIndicador = ri.CodigoIndicador
	ORDER BY
	cf.ValorCampoSomenteLeitura,
	inw.DataTerminoInstancia ASC) as 'NomeProjeto',
tum.SiglaUnidadeMedida AS 'UnidadeMedida',
i.CasasDecimais,
i.Polaridade,
u.CodigoUsuario AS 'Cod_Responsavel_Ind',
u.NomeUsuario AS 'ResponsavelIndicador',
u2.CodigoUsuario AS 'Cod_Responsavel_Coleta',
u2.NomeUsuario AS 'ResponsavelColeta',
'TipoIndicadorEstPro'='E',
(CASE 
        WHEN i.IndicaCriterio = 'S' THEN 'Status'
        WHEN i.IndicaCriterio = 'A' THEN 'Acumulado'
        ELSE 'Avaliar'
    END) AS 'TipoIndicadorStatus',
tfd.NomeFuncao AS 'Agrupamento',
tp.DescricaoPeriodicidade_PT AS 'Periodicidade',
(SELECT TOP 1
ap.Analise
from AnalisePerformance ap
WHERE ap.IndicaTipoIndicador = 'E'
AND ap.DataExclusao IS NULL
AND ap.CodigoIndicador = ri.CodigoIndicador 
AND ap.Ano = ri.Ano 
AND ap.Mes = ri.Mes
) AS 'Analise',
Subconsulta.MetaAcumuladaAno AS 'MetaAcumuladaAno',
ri.MetaMes,
me.TituloMapaEstrategico AS 'MapaEstrategico',
oe.DescricaoObjetoEstrategia AS 'ObjetivoEstrategico',
i.FonteIndicador AS 'Fonte',
i.GlossarioIndicador,
iu.CodigoReservado,
null as 'CodigoMetaOperacional',
i.DataInclusao AS 'DataCriacaoIndicador'
FROM 
    ResumoIndicador ri
LEFT JOIN UnidadeNegocio un ON (un.CodigoUnidadeNegocio = ri.CodigoUnidadeNegocio)
LEFT JOIN Indicador i ON (ri.CodigoIndicador = i.CodigoIndicador)
LEFT JOIN IndicadorUnidade iu ON (iu.CodigoIndicador = i.CodigoIndicador)
LEFT JOIN TipoPeriodicidade tp ON (i.CodigoPeriodicidadeCalculo = tp.CodigoPeriodicidade)
LEFT JOIN TipoUnidadeMedida tum ON (i.CodigoUnidadeMedida = tum.CodigoUnidadeMedida)
LEFT JOIN TipoFuncaoDado tfd ON (i.CodigoFuncaoAgrupamentoMeta = tfd.CodigoFuncao)
LEFT JOIN IndicadorObjetivoEstrategico ioe ON (ioe.CodigoIndicador = ri.CodigoIndicador)
LEFT JOIN ObjetoEstrategia oe ON (oe.codigoobjetoEstrategia = ioe.CodigoObjetivoEstrategico)
LEFT JOIN MapaEstrategico me ON (oe.CodigoMapaEstrategico = me.CodigoMapaEstrategico)
LEFT JOIN Usuario u ON (i.CodigoUsuarioResponsavel = u.CodigoUsuario)
LEFT JOIN Usuario u2 ON (i.CodigoResponsavelAtualizacaoIndicador = u2.CodigoUsuario)

INNER JOIN 
    Subconsulta ON ri.Ano = Subconsulta.Ano 
                AND ri.Mes = Subconsulta.Mes 
                AND ri.CodigoIndicador = Subconsulta.CodigoIndicador 
                AND ri.CodigoUnidadeNegocio = Subconsulta.CodigoUnidadeNegocio

WHERE  un.CodigoUnidadeNegocioSuperior IS NOT NULL
	AND un.DataExclusao IS NULL
	AND iu.DataExclusao IS NULL
	AND oe.DataExclusao IS NULL
	AND i.DataExclusao IS NULL
	AND ri.Mes % tp.IntervaloMeses = 0
ORDER BY
	un.CodigoUnidadeNegocio ASC,
	ri.CodigoIndicador ASC,
	ri.Ano DESC,
	ri.Mes DESC
    """
    cursor.execute(query)

    # Obter os nomes das colunas
    columns = [desc[0] for desc in cursor.description]
    select = cursor.fetchall()
 
    # Desmebra a lista de Tuplas em uma tabela
    tabela_ind_estrategico = pd.DataFrame.from_records(select, columns=columns)

except pyodbc.OperationalError as erro:
    # Captura e trata erros operacionais específicos do pyodbc
    print(f"Erro operacional ao criar a conexão: {erro}")

except Exception as erro:
    # Captura e trata qualquer outro tipo de exceção
    print(f"Erro ao criar a conexão ou executar a consulta: {erro}")

finally:
    # Fecha a conexão com o banco de dados, se estiver aberta
    if cursor:
        cursor.close()
        conexao.close()

# Filtra o DataFrame removendo as linhas que contenham "EXCLUIR" na coluna GlossarioIndicador
tabela_ind_estrategico = tabela_ind_estrategico[~tabela_ind_estrategico['GlossarioIndicador'].str.contains("EXCLUIR", na=False)]

%run Caminhos.ipynb

# salvar os dados em um arquivo parquet
%timeit tabela_ind_estrategico.to_parquet(f'{pasta_hierarquia}{caminho_tabela_ind_estrategico_parquet}')

#!dir "C:/Users/VeryPC/OneDrive - verytecnologia.com.br/4.PIAUÍ - 1.SEDUC/2. Execução/2.5 Projetos/Escritório de Projetos/4. Painéis/Painel iSEDUC/Indicadores_Hierarquia/Query_indicadores/"

Erro ao conectar: ORA-12545: Connect failed because target host or object does not exist
Help: https://docs.oracle.com/error-help/db/ora-12545/
Conexão bem sucedida
Erro ao conectar: ORA-12545: Connect failed because target host or object does not exist
Help: https://docs.oracle.com/error-help/db/ora-12545/
108 ms ± 11.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [2]:
# Conexão e Dataframe 10.10.1.87
# Query Aluno

import pyodbc
import pandas as pd

%run Caminhos.ipynb

dados_conexao_seduc

try:
    # Abrir conexão
    conexao = pyodbc.connect(dados_conexao_seduc)
    print("Conexão bem sucedida")

    # Executa uma consulta
    cursor = conexao.cursor()

    # Executa o Select
    query = """
    SELECT
    a.idAluno,
    a.codigoEscolarAluno,
    a.RA,
    a.codigoAlunoInep,
    a.nome,
    a.cpf,
    a.dataNascimento,
    CASE 
        WHEN a.sexo IS NULL THEN 'Não Declarada'
        ELSE a.sexo 
    END AS 'sexo',
    CASE 
        WHEN a.raca IS NULL THEN 'Não Declarada'
        ELSE a.raca 
    END AS 'raca',
    CASE
        WHEN a.nacionalidade IS NULL THEN 'Não declarada'
        ELSE a.nacionalidade
    END AS 'nacionalidade',
    a.idMunicipio as 'Id Municipio',
    a.logradouroEndereco,
    a.numeroEndereco,
    a.cepEndereco,
    a.bairroEndereco,
    a.idMunicipioEndereco
    FROM 
        aluno a
    """
    cursor.execute(query)

    # Obter os nomes das colunas
    columns = [desc[0] for desc in cursor.description]
    select = cursor.fetchall()

    # Desmebra a lista de Tuplas em uma tabela
    tabela_alunos = pd.DataFrame.from_records(select, columns=columns)

except pyodbc.OperationalError as erro:
    # Captura e trata erros operacionais específicos do pyodbc
    print(f"Erro operacional ao criar a conexão: {erro}")

except Exception as erro:
    # Captura e trata qualquer outro tipo de exceção
    print(f"Erro ao criar a conexão ou executar a consulta: {erro}")

finally:
    # Fecha a conexão com o banco de dados, se estiver aberta
    if cursor:
        cursor.close()
        conexao.close()

%run Caminhos.ipynb

# salvar os dados em um arquivo pickle
#%timeit tabela_alunos.to_pickle(f'{pasta_hierarquia}{caminho_tabela_alunos_pkl}')

# salvar os dados em um arquivo parquet
%timeit tabela_alunos.to_parquet(f'{pasta_hierarquia}{caminho_tabela_alunos_parquet}')

# salvar os dados em um arquivo feather
#%timeit tabela_alunos.to_feather(f'{pasta_hierarquia}{caminho_tabela_alunos_feather}')

#!dir "C:/Users/VeryPC/OneDrive - verytecnologia.com.br/4.PIAUÍ - 1.SEDUC/2. Execução/2.5 Projetos/Escritório de Projetos/4. Painéis/Painel iSEDUC/Indicadores_Hierarquia/Query_indicadores/"

Driver={SQL Server};Server=briskdb.seduc.pi.gov.br,1433;Database=briskdb;UID=usr_briskdb_verytec_bi;PWD=ROcqvkm0Brisk@
Erro ao conectar: ORA-12170: Cannot connect. - timeout of -s for host 10.0.57.18 port 1521. (CONNECTION_ID=5Oo6JztHSySavfKFb0WXhQ==)
Help: https://docs.oracle.com/error-help/db/ora-12170/
Conexão bem sucedida
Driver={SQL Server};Server=briskdb.seduc.pi.gov.br,1433;Database=briskdb;UID=usr_briskdb_verytec_bi;PWD=ROcqvkm0Brisk@
Erro ao conectar: ORA-12170: Cannot connect. - timeout of -s for host 10.0.57.18 port 1521. (CONNECTION_ID=extcF/VUR3eilcAA8xD+JA==)
Help: https://docs.oracle.com/error-help/db/ora-12170/
1.63 s ± 184 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [3]:
# Conexão e Dataframe 10.10.1.58
# Query Indicadores Projetos Operacionais

import pyodbc
import pandas as pd

%run Caminhos.ipynb

dados_conexao_brisk

try:
    # Abrir conexão
    conexao = pyodbc.connect(dados_conexao_brisk)
    print("Conexão bem sucedida")

    # Executa uma consulta
    cursor = conexao.cursor()

    # Executa o Select
    query = """
SELECT 
    un.CodigoUnidadeNegocio,
    un.CodigoUnidadeNegocioSuperior,
    un.SiglaUnidadeNegocio AS 'SiglaUN',
    un.NomeUnidadeNegocio AS 'NomeUnidade',
    un.CorUnidadeNegocio,
    io.CodigoIndicador as 'Codigo Indicador',
    io.NomeIndicador AS 'Indicador',
    rmo.Ano as 'Ano_Meta',
    rmo.Mes as 'Mes_Meta',
    p.CodigoProjeto,
    p.NomeProjeto,
    tum.SiglaUnidadeMedida AS 'UnidadeMedida',
    io.CasasDecimais,
    io.Polaridade,
    u.CodigoUsuario AS 'Cod_Responsavel_Ind',
    u.NomeUsuario AS 'ResponsavelIndicador',
    u2.CodigoUsuario AS 'Cod_Responsavel_Coleta',
    u2.NomeUsuario AS 'ResponsavelColeta',
    io.TipoIndicador AS 'TipoIndicadorEstPro',
    CASE 
        WHEN io.IndicaCriterio = 'S' THEN 'Status'
        WHEN io.IndicaCriterio = 'A' THEN 'Acumulado'
        ELSE 'Avaliar'
    END AS 'TipoIndicadorStatus',
    tfd.NomeFuncao AS 'Agrupamento',
    tp.DescricaoPeriodicidade_PT AS 'Periodicidade',

    -- Análise de Performance
    (
        SELECT TOP 1 ap.Analise
        FROM AnalisePerformance ap
        WHERE ap.IndicaTipoIndicador IN ('O', 'D')
            AND ap.DataExclusao IS NULL
            AND ap.CodigoIndicador = io.CodigoIndicador 
            AND ap.CodigoObjetoAssociado = mo.CodigoMetaOperacional
            AND ap.Ano = rmo.Ano 
            AND ap.Mes = rmo.Mes 
        ORDER BY ap.DataAnalisePerformance
    ) AS 'Analise',

    rmo.ValorMetaAcumuladaAno AS 'MetaAcumuladaAno',
    rmo.ValorMetaPeriodo AS 'MetaMes',

    -- Mapa Estratégico (corrigido com TOP 1)
    (
        SELECT TOP 1 me.TituloMapaEstrategico    
        FROM ObjetoEstrategia oe
        LEFT JOIN ProjetoObjetivoEstrategico poe ON poe.CodigoProjeto = p.CodigoProjeto
        LEFT JOIN MapaEstrategico me ON oe.CodigoMapaEstrategico = me.CodigoMapaEstrategico
        WHERE p.CodigoProjeto = poe.CodigoProjeto
          AND poe.CodigoObjetivoEstrategico = oe.CodigoObjetoEstrategia
          AND oe.DataExclusao IS NULL
        ORDER BY me.TituloMapaEstrategico
    ) AS 'MapaEstrategico',

    -- Objetivo Estratégico
    CASE 
        WHEN (
            SELECT COUNT(poe.CodigoProjeto)
            FROM ProjetoObjetivoEstrategico poe
            LEFT JOIN ObjetoEstrategia oe ON oe.CodigoObjetoEstrategia = poe.CodigoObjetivoEstrategico
            WHERE p.CodigoProjeto = poe.CodigoProjeto
              AND oe.DataExclusao IS NULL
        ) > 1 THEN 'DUPLICADO'
        ELSE (
            SELECT TOP 1 oe.DescricaoObjetoEstrategia
            FROM ObjetoEstrategia oe
            LEFT JOIN ProjetoObjetivoEstrategico poe ON poe.CodigoProjeto = p.CodigoProjeto
            LEFT JOIN MapaEstrategico me ON oe.CodigoMapaEstrategico = me.CodigoMapaEstrategico
            WHERE oe.DataExclusao IS NULL
              AND p.CodigoProjeto = poe.CodigoProjeto
              AND poe.CodigoObjetivoEstrategico = oe.CodigoObjetoEstrategia
            ORDER BY oe.DescricaoObjetoEstrategia
        )
    END AS 'ObjetivoEstrategico',

    mo.FonteIndicador AS 'Fonte',
    io.GlossarioIndicador,
    NULL AS 'CodigoReservado',
    mo.CodigoMetaOperacional,
    io.DataInclusao AS 'DataCriacaoIndicador'

FROM IndicadorOperacional io 
LEFT JOIN TipoUnidadeMedida tum ON tum.CodigoUnidadeMedida = io.CodigoUnidadeMedida
LEFT JOIN Usuario u ON u.CodigoUsuario = io.CodigoUsuarioResponsavel
LEFT JOIN TipoFuncaoDado tfd ON tfd.CodigoFuncao = io.CodigoFuncaoAgrupamentoMeta
LEFT JOIN MetaOperacional mo ON mo.CodigoIndicador = io.CodigoIndicador
LEFT JOIN TipoPeriodicidade tp ON tp.CodigoPeriodicidade = mo.CodigoPeriodicidade
LEFT JOIN Usuario u2 ON u2.CodigoUsuario = mo.CodigoUsuarioResponsavelAtualizacao
LEFT JOIN Projeto p ON p.CodigoProjeto = mo.CodigoProjeto
LEFT JOIN UnidadeNegocio un ON p.CodigoUnidadeNegocio = un.CodigoUnidadeNegocio
LEFT JOIN ResumoMetaOperacional rmo ON rmo.CodigoMetaOperacional = mo.CodigoMetaOperacional
LEFT JOIN TipoProjeto tpro ON tpro.CodigoTipoProjeto = p.CodigoTipoProjeto

WHERE io.DataExclusao IS NULL
  AND un.DataExclusao IS NULL
  AND (p.CodigoTipoProjeto = 1 OR p.CodigoTipoProjeto = 2)
  AND p.DataExclusao IS NULL

ORDER BY 
    un.CodigoUnidadeNegocio ASC,
    io.CodigoIndicador ASC,
    rmo.Ano DESC,
    rmo.Mes DESC
    """
    cursor.execute(query)

    # Obter os nomes das colunas
    columns = [desc[0] for desc in cursor.description]
    select = cursor.fetchall()
 
    # Desmebra a lista de Tuplas em uma tabela
    tabela_indicadores_projetos = pd.DataFrame.from_records(select, columns=columns)

except pyodbc.OperationalError as erro:
    # Captura e trata erros operacionais específicos do pyodbc
    print(f"Erro operacional ao criar a conexão: {erro}")

except Exception as erro:
    # Captura e trata qualquer outro tipo de exceção
    print(f"Erro ao criar a conexão ou executar a consulta: {erro}")

finally:
    # Fecha a conexão com o banco de dados, se estiver aberta
    if cursor:
        cursor.close()
        conexao.close()

# Filtra o DataFrame removendo as linhas que contenham "EXCLUIR" na coluna GlossarioIndicador
tabela_indicadores_projetos = tabela_indicadores_projetos[~tabela_indicadores_projetos['GlossarioIndicador'].str.contains("EXCLUIR", na=False)]

%run Caminhos.ipynb

# salvar os dados em um arquivo parquet
%timeit tabela_indicadores_projetos.to_parquet(f'{pasta_hierarquia}{caminho_tabela_indicadores_projetos_parquet}')

#!dir "C:/Users/VeryPC/OneDrive - verytecnologia.com.br/4.PIAUÍ - 1.SEDUC/2. Execução/2.5 Projetos/Escritório de Projetos/4. Painéis/Painel iSEDUC/Indicadores_Hierarquia/Query_indicadores/"

Erro ao conectar: ORA-12545: Connect failed because target host or object does not exist
Help: https://docs.oracle.com/error-help/db/ora-12545/
Conexão bem sucedida
Erro ao conectar: ORA-12545: Connect failed because target host or object does not exist
Help: https://docs.oracle.com/error-help/db/ora-12545/
21.5 ms ± 557 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [3]:
# Base Gerada com os dados inputados manualmente na planilha Base_Unificada_Indicadores

import pandas as pd
from openpyxl import load_workbook

# Carregar o arquivo Excel
BASE_GERADA = load_workbook("C:/Users/VeryPC/OneDrive - verytecnologia.com.br/4.PIAUÍ - 1.SEDUC/2. Execução/2.5 Projetos/Escritório de Projetos/4. Painéis/Painel iSEDUC/Indicadores_Hierarquia/Base_Unificada_Indicadores.xlsm", data_only=True)

# Acessar a aba específica
aba_especifica = BASE_GERADA["BASE_GERADA"]
    
# Localizar a tabela "TAB_META"
tabela = aba_especifica.tables["BASE_GERADA"]

# Obter o intervalo da tabela
dados_tabela = aba_especifica[tabela.ref]

# Extrair os dados da tabela
dados = [[cell.value for cell in row] for row in dados_tabela]

# Criar o DataFrame
colunas = dados[0]  # Assumindo que a primeira linha contém os nomes das colunas
BASE_GERADA = pd.DataFrame(dados[1:], columns=colunas)
BASE_GERADA = BASE_GERADA.drop(['Editado'], axis=1)

# Função para extrair a parte não numérica
def extrair_nao_numerico(codigo):
    codigo = str(codigo)  # Converte para string
    for char in codigo:
        if not char.isdigit():
            return codigo[codigo.index(char):]  # Retorna a parte não numérica
    return ""  # Retorna string vazia se todos os caracteres forem números

# Criando a nova coluna
BASE_GERADA['Codigo Extraido'] = BASE_GERADA['Codigo Indicador'].apply(extrair_nao_numerico)

# Função para remover a parte não numérica do 'Codigo Indicador'
def remover_codigo_extraido(codigo_indicador, codigo_extraido):
    if codigo_extraido:
        return codigo_indicador.replace(codigo_extraido, "")
    return codigo_indicador

# Aplicar a função para remover a parte não numérica do 'Codigo Indicador'
BASE_GERADA['Codigo Indicador'] = BASE_GERADA.apply(lambda row: remover_codigo_extraido(str(row['Codigo Indicador']), row['Codigo Extraido']), axis=1)
# Remover a coluna 'Codigo Extraido'
BASE_GERADA.drop(columns=['Codigo Extraido'], inplace=True)

#---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
# Mesclar pra pega a coluna Tipo de Indicador
%run Caminhos.ipynb
tabela_especificacao_ind_O_E = pd.read_parquet(f'{pasta_hierarquia}{caminho_especificacao_ind_O_E_parquet}')
tabela_especificacao_ind_O_E = tabela_especificacao_ind_O_E[['Codigo Indicador', 'Indicador', 'TipoIndicadorEstPro']]
tabela_especificacao_ind_O_E['Codigo Indicador'] = tabela_especificacao_ind_O_E['Codigo Indicador'].astype(str)
tabela_especificacao_ind_O_E['Indicador'] = tabela_especificacao_ind_O_E['Indicador'].astype(str)

# Removendo duplicatas com base nas colunas selecionadas
tabela_especificacao_ind_O_E = tabela_especificacao_ind_O_E.drop_duplicates()

# Concatenar as colunas usando o operador +
tabela_especificacao_ind_O_E['KEY'] = tabela_especificacao_ind_O_E['Codigo Indicador'] + tabela_especificacao_ind_O_E['Indicador']
#tabela_especificacao_ind_O_E.info()
# Concatenar as colunas usando o operador +
BASE_GERADA['KEY'] = BASE_GERADA['Codigo Indicador'] + BASE_GERADA['Indicador']

# Mesclar os DataFrames mantendo apenas a coluna 'TipoIndicadorEstPro' de df2
BASE_GERADA = BASE_GERADA.merge(tabela_especificacao_ind_O_E[['KEY', 'TipoIndicadorEstPro']], on='KEY', how='left')
#---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Remover a coluna 'Codigo Extraido'
BASE_GERADA.drop(columns=['KEY'], inplace=True)

# Aplicar a condição
BASE_GERADA['Quantidade'] = BASE_GERADA.apply(lambda row: row['Quantidade'] / 100 if row['UnidadeMedida'] == '%' else row['Quantidade'], axis=1)
#BASE_GERADA = BASE_GERADA[BASE_GERADA['Codigo Indicador'] == "2216"]
#BASE_GERADA.info()
# display(BASE_GERADA)

# Verificar o tipo da coluna 'Data Cadastro'
#print(BASE_GERADA['Data Cadastro'].dtype)

# Converter a coluna 'Data Cadastro' para o formato de string ou datetime apropriado
BASE_GERADA['Data Cadastro'] = pd.to_datetime(BASE_GERADA['Data Cadastro'], errors='coerce')
#display(BASE_GERADA)
# salvar os dados em um arquivo pickle
#%timeit BASE_GERADA.to_pickle(f'{pasta_hierarquia}{caminho_tabela_base_gerada_pkl}')
display(BASE_GERADA)
# salvar os dados em um arquivo parquet
%timeit BASE_GERADA.to_parquet(f'{pasta_hierarquia}{caminho_tabela_base_gerada_parquet}')

# salvar os dados em um arquivo csv
%timeit BASE_GERADA.to_csv(f'{pasta_hierarquia}{caminho_teste}')

# salvar os dados em um arquivo feather
#%timeit BASE_GERADA.to_feather(f'{pasta_hierarquia}{caminho_tabela_base_gerada_feather}')

#!dir "C:/Users/VeryPC/OneDrive - verytecnologia.com.br/4.PIAUÍ - 1.SEDUC/2. Execução/2.5 Projetos/Escritório de Projetos/4. Painéis/Painel iSEDUC/Indicadores_Hierarquia/Query_indicadores/"

c:\Users\VeryPC\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Erro ao conectar: ORA-12545: Connect failed because target host or object does not exist
Help: https://docs.oracle.com/error-help/db/ora-12545/


C:\Users\VeryPC\AppData\Local\Temp\ipykernel_24504\3209263483.py:82: UserWarning: Parsing dates in %d/%m/%Y  %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  BASE_GERADA['Data Cadastro'] = pd.to_datetime(BASE_GERADA['Data Cadastro'], errors='coerce')


,Codigo Indicador,Indicador,Ano,Mes_Referente,Pais,Regiao,Id Estado,Estado,Territorio Desenvolv,Id Ger. Regional,...,Id Escola,Escola,Resultado Comparativo,Localizacao,Situacao,Quantidade,UnidadeMedida,Hierarquia,Data Cadastro,TipoIndicadorEstPro
0,415,% de municípios que assinaram a adesão ao Pact...,2023,6,Brasil,Nordeste,22.0,PIAUÍ,None,NaN,...,NaN,None,NÃO,None,None,0.5600,%,Estado,2024-08-27 16:52:29,O
1,2221,Taxa de escolarização da população de 4 a 5 anos,2023,12,Brasil,Nordeste,22.0,PIAUÍ,None,NaN,...,NaN,None,NÃO,None,None,0.9880,%,Estado,2024-08-27 12:06:56,E
2,2220,Taxa de escolarização da população de 0 a 3 anos,2022,12,Brasil,Nordeste,22.0,PIAUÍ,None,NaN,...,NaN,None,NÃO,None,None,0.3160,%,Estado,2024-08-27 12:06:56,E
3,2220,Taxa de escolarização da população de 0 a 3 anos,2023,12,Brasil,Nordeste,22.0,PIAUÍ,None,NaN,...,NaN,None,NÃO,None,None,0.3690,%,Estado,2024-08-27 12:06:56,E
4,2251,Proficiência em Língua Portuguesa dos Estudant...,2023,12,Brasil,Nordeste,22.0,PIAUÍ,None,NaN,...,NaN,None,NÃO,None,None,6.2582,%,Estado,2024-08-27 12:34:41,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27775,2255,Proficiência Média em Matemática dos Estudante...,2024,12,Brasil,Nordeste,22.0,PIAUÍ,VALE DO SAMBITO,320.0,...,NaN,None,None,None,None,498.0000,Nº,Municipio,2025-06-02 10:41:28,E
27776,2255,Proficiência Média em Matemática dos Estudante...,2024,12,Brasil,Nordeste,22.0,PIAUÍ,VALE DO RIO GUARIBAS,398.0,...,NaN,None,None,None,None,506.0000,Nº,Municipio,2025-06-02 10:41:28,E
27777,2255,Proficiência Média em Matemática dos Estudante...,2024,12,Brasil,Nordeste,22.0,PIAUÍ,SERRA DA CAPIVARA,306.0,...,NaN,None,None,None,None,520.0000,Nº,Municipio,2025-06-02 10:41:28,E
27778,2255,Proficiência Média em Matemática dos Estudante...,2024,12,Brasil,Nordeste,22.0,PIAUÍ,VALE DOS RIOS PIAUÍ E ITAUEIRA,337.0,...,NaN,None,None,None,None,615.0000,Nº,Municipio,2025-06-02 10:41:28,E


54.7 ms ± 3.66 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
265 ms ± 49.7 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [5]:
# Query Laboratório

import pandas as pd
import cx_Oracle

# Definir as credenciais e informações de conexão
dsn_tns = cx_Oracle.makedsn('sc01-c1-scan.ati.pi.gov.br', '1521', service_name='SEDUCP')
conexao = cx_Oracle.connect(user='data_estados', password='data_estados', dsn=dsn_tns)

# Executa uma consulta
cursor = conexao.cursor()

query = """
SELECT
E.NU_ANO_CENSO as "Ano",
E.CO_ENTIDADE AS "Id Escola",
E.IN_LABORATORIO_INFORMATICA AS LABORATORIO_INFORMATICA, /* COUNT(DISTINCT CASE WHEN E.IN_LABORATORIO_INFORMATICA = 1 THEN E.CO_ENTIDADE END) * 1.0 / COUNT(DISTINCT E.CO_ENTIDADE) AS PROPORCAO_LABORATORIO */
E.IN_LABORATORIO_CIENCIAS AS LABORATORIO_CIENCIAS, /* COUNT(DISTINCT CASE WHEN E.IN_LABORATORIO_CIENCIAS = 1 THEN E.CO_ENTIDADE END) * 1.0 / COUNT(DISTINCT E.CO_ENTIDADE) AS PROPORCAO_LABORATORIO_CIENCIA */
E.IN_SALA_ATENDIMENTO_ESPECIAL AS SALA_ATENDIMENTO_ESPECIAL /* COUNT(DISTINCT CASE WHEN E.IN_SALA_ATENDIMENTO_ESPECIAL = 1 THEN E.CO_ENTIDADE END) * 1.0 / COUNT(DISTINCT E.CO_ENTIDADE) AS PROPORCAO_ATENDIMENTO_AEE */
FROM 
    TS_ESCOLA_2023 E
WHERE
    E.NU_ANO_CENSO = 2023
    AND E.CO_UF = 22
    AND E.TP_DEPENDENCIA = 2
    AND E.TP_SITUACAO_FUNCIONAMENTO = 1
"""
cursor.execute(query)

indicador_censo = pd.read_sql(query, conexao)

# Estabelecer a conexão
print("Conexão bem sucedida")
display(indicador_censo)

%run Caminhos.ipynb

# salvar os dados em um arquivo pickle
%timeit indicador_censo.to_parquet(f'{pasta_hierarquia}{caminho_indicador_censo_parquet}')

Conexão bem sucedida


C:\Users\VeryPC\AppData\Local\Temp\ipykernel_22408\136154777.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  indicador_censo = pd.read_sql(query, conexao)


,Ano,Id Escola,LABORATORIO_INFORMATICA,LABORATORIO_CIENCIAS,SALA_ATENDIMENTO_ESPECIAL
0,2023,22026983,1,0,1
1,2023,22027629,1,0,1
2,2023,22045686,1,1,1
3,2023,22046046,0,0,0
4,2023,22223606,1,0,0
...,...,...,...,...,...
635,2023,22028315,1,0,1
636,2023,22124586,0,0,0
637,2023,22048340,1,1,1
638,2023,22050450,1,1,1


Driver={SQL Server};Server=briskdb.seduc.pi.gov.br,1433;Database=briskdb;UID=usr_briskdb_verytec_bi;PWD=ROcqvkm0Brisk@
Conexão bem-sucedida!
3.76 ms ± 581 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [4]:
# Query Matricutla Oracle

import pandas as pd
import cx_Oracle

# Definir as credenciais e informações de conexão
dsn_tns = cx_Oracle.makedsn('sc01-c1-scan.ati.pi.gov.br', '1521', service_name='SEDUCP')
conexao = cx_Oracle.connect(user='data_estados', password='data_estados', dsn=dsn_tns)

# Executa uma consulta
cursor = conexao.cursor()

query = """
SELECT
E.NU_ANO_CENSO as "Ano",
E.CO_ENTIDADE AS "Id Escola",
M.ID_MATRICULA,
E.TP_SITUACAO_FUNCIONAMENTO,
AE.TP_ETAPA_ENSINO,
AE.NO_ETAPA_ENSINO
FROM TS_MATRICULA_2023 M
LEFT JOIN TS_ESCOLA_2023 E ON E.CO_ENTIDADE = M.CO_ENTIDADE
LEFT JOIN AUX_ETAPA_ENSINO_ISCED AE ON AE.TP_ETAPA_ENSINO = M.TP_ETAPA_ENSINO
WHERE M.NU_ANO_CENSO = 2023
AND M.CO_UF = 22
AND M.TP_DEPENDENCIA = 2
"""
cursor.execute(query)

indicador_censo_matriculas = pd.read_sql(query, conexao)

# Estabelecer a conexão
print("Conexão bem sucedida")
display(indicador_censo_matriculas)

%run Caminhos.ipynb

# salvar os dados em um arquivo pickle
%timeit indicador_censo_matriculas.to_parquet(f'{pasta_hierarquia}{caminho_indicador_censo_matriculas_parquet}')

C:\Users\VeryPC\AppData\Local\Temp\ipykernel_22408\672478534.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  indicador_censo_matriculas = pd.read_sql(query, conexao)


Conexão bem sucedida


,Ano,Id Escola,ID_MATRICULA,TP_SITUACAO_FUNCIONAMENTO,TP_ETAPA_ENSINO,NO_ETAPA_ENSINO
0,2023,22135880,666029623,1,70.0,EJA - Ensino Fundamental - Anos finais
1,2023,22135880,666029623,1,70.0,EJA - Ensino Fundamental - Anos finais
2,2023,22135880,666029623,1,70.0,EJA - Ensino Fundamental - Anos finais
3,2023,22135880,666029623,1,70.0,EJA - Ensino Fundamental - Anos finais
4,2023,22135880,666029623,1,70.0,EJA - Ensino Fundamental - Anos finais
...,...,...,...,...,...,...
3225465,2023,22048928,665270499,1,25.0,Ensino Médio - 1ª Série
3225466,2023,22048928,665270499,1,25.0,Ensino Médio - 1ª Série
3225467,2023,22048928,665270499,1,25.0,Ensino Médio - 1ª Série
3225468,2023,22048928,665270499,1,25.0,Ensino Médio - 1ª Série


Driver={SQL Server};Server=briskdb.seduc.pi.gov.br,1433;Database=briskdb;UID=usr_briskdb_verytec_bi;PWD=ROcqvkm0Brisk@
Conexão bem-sucedida!
912 ms ± 51.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
# Conexão e Dataframe 10.10.1.87
# Query Professor

import pyodbc
import pandas as pd

%run Caminhos.ipynb

dados_conexao_seduc

try:
    # Abrir conexão
    conexao = pyodbc.connect(dados_conexao_seduc)
    print("Conexão bem sucedida")

    # Executa uma consulta
    cursor = conexao.cursor()

    # Executa o Select
    query = """
    SELECT
    lo.idLotacao,
    lo.idPeriodo,
    pl.anoLetivo,
    lo.nome,
    lo.idGerenciaRegional AS "Gerencia Regional",
    lo.idEscola AS "Id Escola",
    lo.idVinculo,
    v.idServidor,
    v.matricula,
    v.nomeServidor,
    v.codigoServidorInep,
    v.cpf,
    v.dataNascimento,
    v.sexo,
    v.especialidade,
    v.grauInstrucao,
    v.formacaoProfissional,
    t.idTurma,
    t.nome AS "Nome_turma",
    t.ativa AS "Turma_Ativa",
    t.finalizada,
    lo.idTurmaDisciplina,
    lo.funcao,
    lo.cargo,
    lo.ativa
    FROM 
        lotacao lo
    LEFT JOIN 
        turmaDisciplina td on lo.idTurmaDisciplina = td.idTurmaDisciplina
    LEFT JOIN 
        turma t on t.idTurma = td.idTurma
    LEFT JOIN
        periodoLetivo pl on pl.idPeriodo = lo.idPeriodo
    LEFT JOIN
        vinculo v ON v.idVinculo = lo.idVinculo
    """
    cursor.execute(query)

    # Obter os nomes das colunas
    columns = [desc[0] for desc in cursor.description]
    select = cursor.fetchall()

    # Desmebra a lista de Tuplas em uma tabela
    tabela_professor = pd.DataFrame.from_records(select, columns=columns)

except pyodbc.OperationalError as erro:
    # Captura e trata erros operacionais específicos do pyodbc
    print(f"Erro operacional ao criar a conexão: {erro}")

except Exception as erro:
    # Captura e trata qualquer outro tipo de exceção
    print(f"Erro ao criar a conexão ou executar a consulta: {erro}")

finally:
    # Fecha a conexão com o banco de dados, se estiver aberta
    if cursor:
        cursor.close()
        conexao.close()

%run Caminhos.ipynb

# salvar os dados em um arquivo pickle
#%timeit tabela_professor.to_pickle(f'{pasta_hierarquia}{caminho_tabela_professor_pkl}')

# salvar os dados em um arquivo parquet
%timeit tabela_professor.to_parquet(f'{pasta_hierarquia}{caminho_tabela_professor_parquet}')

# salvar os dados em um arquivo feather
#%timeit tabela_professor.to_feather(f'{pasta_hierarquia}{caminho_tabela_professor_feather}')

#!dir "C:/Users/VeryPC/OneDrive - verytecnologia.com.br/4.PIAUÍ - 1.SEDUC/2. Execução/2.5 Projetos/Escritório de Projetos/4. Painéis/Painel iSEDUC/Indicadores_Hierarquia/Query_indicadores/"

Conexão bem sucedida
1.56 s ± 28.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
# Conexão e Dataframe 10.10.1.87
# Query Nota

import pyodbc
import pandas as pd

%run Caminhos.ipynb

dados_conexao_seduc

try:
    # Abrir conexão
    conexao = pyodbc.connect(dados_conexao_seduc)
    print("Conexão bem sucedida")

    # Executa uma consulta
    cursor = conexao.cursor()

    # Executa o Select
    query = """
    SELECT
    m.idEtapa,
    e.nome AS nomeEtapa,
    e.agregada,
    e.modalidadeEnsinoISeduc,
    est.idEstado as 'Id Estado',
    est.nome as 'Estado',
    mu.territorioDesenvolvimento as 'Territorio Desenvolv', 
    gr.nome as 'Gerencia Regional', 
    CAST(mu.idMunicipio as INT) as 'Id Municipio', 
    mu.nome as 'Municipio',
    es.idEscola as 'Id Escola',
    m.idMatricula,
    d.idDisciplina,
    d.nome AS nomeDisciplina,
    tn.idTipoNota,
    tn.nome AS nomeTipoNota,
    tn.descricao,
    tn.grupo,
    AVG(n.valorNota) AS mediaNota,
    CAST(pl.anoLetivo AS INT) as 'Ano'
    FROM matricula m 
    LEFT JOIN turma t ON t.idTurma = m.idTurma
    LEFT JOIN turmaDisciplina td ON td.idTurma = t.idTurma
    LEFT JOIN disciplina d ON d.idDisciplina = td.idDisciplina
    LEFT JOIN nota n ON n.idTurmaDisciplina = td.idTurmaDisciplina AND n.idMatricula = m.idMatricula
    LEFT JOIN tipoNota tn ON tn.idTipoNota = n.idTipoNota
    LEFT JOIN periodoLetivo pl ON pl.idPeriodo = m.idPeriodo
    LEFT JOIN etapa e ON e.idEtapa = m.idEtapa
    LEFT JOIN etapaTipoNota tne ON tne.idEtapa = e.idEtapa AND tne.idTipoNota = tn.idTipoNota
    LEFT JOIN escola es ON es.idEscola = m.idEscola
    LEFT JOIN municipio mu ON mu.idMunicipio = es.idMunicipio
    LEFT JOIN gerenciaRegional gr ON gr.idGerenciaRegional = es.idGerenciaRegional
    LEFT JOIN estado est ON est.idEstado = mu.idEstado
    WHERE es.dependenciaAdministrativa = 'ESTADUAL'
    AND pl.anoLetivo IN (2024, 2025)
    AND tn.descricao LIKE '%Média%'
    AND tn.nome  LIKE '%MTF%'
    AND m.matriculaAtiva = 1
    AND m.situacaoMatricula = 'Cursando'
    GROUP BY 
        m.idEtapa, e.nome, e.agregada, e.modalidadeEnsinoISeduc, est.idEstado, est.nome, mu.territorioDesenvolvimento, gr.nome, mu.idMunicipio, mu.nome, m.idMatricula, 
        d.idDisciplina, d.nome, tn.idTipoNota, tn.nome, tn.descricao, tn.grupo, es.idEscola, pl.anoLetivo
    ORDER BY m.idMatricula, d.idDisciplina, tn.idTipoNota;
    """
    cursor.execute(query)

    # Obter os nomes das colunas
    columns = [desc[0] for desc in cursor.description]
    select = cursor.fetchall()

    # Desmebra a lista de Tuplas em uma tabela
    tabela_notas = pd.DataFrame.from_records(select, columns=columns)

    # Adicionar uma nova coluna com base na condição 2026
    tabela_notas['Matricula_2026'] = tabela_notas.apply(
        lambda row: 1 if row['Ano'] == 2026 else 0,
        axis=1
    )

    # Adicionar uma nova coluna com base na condição 2025
    tabela_notas['Matricula_2025'] = tabela_notas.apply(
        lambda row: 1 if row['Ano'] == 2025 else 0,
        axis=1
    )

    # Adicionar uma nova coluna com base na condição 2024
    tabela_notas['Matricula_2024'] = tabela_notas.apply(
        lambda row: 1 if row['Ano'] == 2024 else 0,
        axis=1
    )

except pyodbc.OperationalError as erro:
    # Captura e trata erros operacionais específicos do pyodbc
    print(f"Erro operacional ao criar a conexão: {erro}")

except Exception as erro:
    # Captura e trata qualquer outro tipo de exceção
    print(f"Erro ao criar a conexão ou executar a consulta: {erro}")

finally:
    # Fecha a conexão com o banco de dados, se estiver aberta
    if cursor:
        cursor.close()
        conexao.close()

%run Caminhos.ipynb

# salvar os dados em um arquivo pickle
#%timeit tabela_notas.to_pickle(f'{pasta_hierarquia}{caminho_tabela_notas_pkl}')

# salvar os dados em um arquivo parquet
%timeit tabela_notas.to_parquet(f'{pasta_hierarquia}{caminho_tabela_notas_parquet}')

# salvar os dados em um arquivo feather
#%timeit tabela_notas.to_feather(f'{pasta_hierarquia}{caminho_tabela_notas_feather}')

#!dir "C:/Users/VeryPC/OneDrive - verytecnologia.com.br/4.PIAUÍ - 1.SEDUC/2. Execução/2.5 Projetos/Escritório de Projetos/4. Painéis/Painel iSEDUC/Indicadores_Hierarquia/Query_indicadores/"

Conexão bem sucedida
9.07 s ± 606 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
# Conexão e Dataframe 10.10.1.87
# Query Nota Geral

import pyodbc
import pandas as pd

%run Caminhos.ipynb

dados_conexao_seduc

try:
    # Abrir conexão
    conexao = pyodbc.connect(dados_conexao_seduc)
    print("Conexão bem sucedida")

    # Executa uma consulta
    cursor = conexao.cursor()

    # Executa o Select
    query = """
    SELECT
    m.idEtapa,
    e.nome AS nomeEtapa,
    e.agregada,
    e.modalidadeEnsinoISeduc,
    est.idEstado as 'Id Estado',
    est.nome as 'Estado',
    mu.territorioDesenvolvimento as 'Territorio Desenvolv', 
    gr.nome as 'Gerencia Regional', 
    CAST(mu.idMunicipio as INT) as 'Id Municipio', 
    mu.nome as 'Municipio', 
    es.idEscola as 'Id Escola',
    m.idMatricula,
    t.idTurma,
    d.idDisciplina,
    d.nome AS nomeDisciplina,
    tn.idTipoNota,
    tn.nome AS nomeTipoNota,
    tn.descricao,
    tn.grupo,
    n.valorNota,
    CAST(pl.anoLetivo AS INT) as 'Ano'
    FROM matricula m 
    LEFT JOIN turma t ON t.idTurma = m.idTurma
    LEFT JOIN turmaDisciplina td ON td.idTurma = t.idTurma
    LEFT JOIN disciplina d ON d.idDisciplina = td.idDisciplina
    LEFT JOIN nota n ON n.idTurmaDisciplina = td.idTurmaDisciplina AND n.idMatricula = m.idMatricula
    LEFT JOIN tipoNota tn ON tn.idTipoNota = n.idTipoNota
    LEFT JOIN periodoLetivo pl ON pl.idPeriodo = m.idPeriodo
    LEFT JOIN etapa e ON e.idEtapa = m.idEtapa
    LEFT JOIN etapaTipoNota tne ON tne.idEtapa = e.idEtapa AND tne.idTipoNota = tn.idTipoNota
    LEFT JOIN escola es ON es.idEscola = m.idEscola
    LEFT JOIN municipio mu ON mu.idMunicipio = es.idMunicipio
    LEFT JOIN gerenciaRegional gr ON gr.idGerenciaRegional = es.idGerenciaRegional
    LEFT JOIN estado est ON est.idEstado = mu.idEstado
    WHERE es.dependenciaAdministrativa = 'ESTADUAL'
    AND pl.anoLetivo IN (2024, 2025)
    AND tn.descricao LIKE '%Média%'
    AND tn.nome  LIKE '%MTF%'
    AND m.matriculaAtiva = 1
    AND m.situacaoMatricula = 'Cursando'
    ORDER BY m.idMatricula, d.idDisciplina, tn.idTipoNota;
    """
    cursor.execute(query)

    # Obter os nomes das colunas
    columns = [desc[0] for desc in cursor.description]
    select = cursor.fetchall()

    # Desmebra a lista de Tuplas em uma tabela
    tabela_notas_geral = pd.DataFrame.from_records(select, columns=columns)

    # Adicionar uma nova coluna com base na condição 2026
    tabela_notas_geral['Matricula_2026'] = tabela_notas_geral.apply(
        lambda row: 1 if row['Ano'] == 2026 else 0,
        axis=1
    )

    # Adicionar uma nova coluna com base na condição 2025
    tabela_notas_geral['Matricula_2025'] = tabela_notas_geral.apply(
        lambda row: 1 if row['Ano'] == 2025 else 0,
        axis=1
    )

    # Adicionar uma nova coluna com base na condição 2024
    tabela_notas_geral['Matricula_2024'] = tabela_notas_geral.apply(
        lambda row: 1 if row['Ano'] == 2024 else 0,
        axis=1
    )

except pyodbc.OperationalError as erro:
    # Captura e trata erros operacionais específicos do pyodbc
    print(f"Erro operacional ao criar a conexão: {erro}")

except Exception as erro:
    # Captura e trata qualquer outro tipo de exceção
    print(f"Erro ao criar a conexão ou executar a consulta: {erro}")

finally:
    # Fecha a conexão com o banco de dados, se estiver aberta
    if cursor:
        cursor.close()
        conexao.close()

display(tabela_notas_geral)
%run Caminhos.ipynb

# salvar os dados em um arquivo pickle
#%timeit tabela_notas_geral.to_pickle(f'{pasta_hierarquia}{caminho_tabela_notas_geral_pkl}')

# salvar os dados em um arquivo parquet
%timeit tabela_notas_geral.to_parquet(f'{pasta_hierarquia}{caminho_tabela_notas_geral_parquet}')

# salvar os dados em um arquivo feather
#%timeit tabela_notas_geral.to_feather(f'{pasta_hierarquia}{caminho_tabela_notas_geral_feather}')

#!dir "C:/Users/VeryPC/OneDrive - verytecnologia.com.br/4.PIAUÍ - 1.SEDUC/2. Execução/2.5 Projetos/Escritório de Projetos/4. Painéis/Painel iSEDUC/Indicadores_Hierarquia/Query_indicadores/"

Conexão bem sucedida


,idEtapa,nomeEtapa,agregada,modalidadeEnsinoISeduc,Id Estado,Estado,Territorio Desenvolv,Gerencia Regional,Id Municipio,Municipio,...,nomeDisciplina,idTipoNota,nomeTipoNota,descricao,grupo,valorNota,Ano,Matricula_2026,Matricula_2025,Matricula_2024
0,12,Ensino médio - 3ª Série,Ensino Médio,EDUCAÇÃO REGULAR - ENSINO MÉDIO,22,PIAUÍ,VALE DOS RIOS PIAUÍ E ITAUEIRA,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2202091,CALDEIRAO GRANDE DO PIAUI,...,LÍNGUA PORTUGUESA,354,1MTF,MÉDIA TRIMESTRAL FINAL,RESULTADO DO 1º TRIMESTRE,7.5,2024,0,0,1
1,12,Ensino médio - 3ª Série,Ensino Médio,EDUCAÇÃO REGULAR - ENSINO MÉDIO,22,PIAUÍ,VALE DOS RIOS PIAUÍ E ITAUEIRA,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2202091,CALDEIRAO GRANDE DO PIAUI,...,LÍNGUA PORTUGUESA,361,2MTF,MÉDIA TRIMESTRAL FINAL,RESULTADO DO 2º TRIMESTRE,7.8,2024,0,0,1
2,12,Ensino médio - 3ª Série,Ensino Médio,EDUCAÇÃO REGULAR - ENSINO MÉDIO,22,PIAUÍ,VALE DOS RIOS PIAUÍ E ITAUEIRA,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2202091,CALDEIRAO GRANDE DO PIAUI,...,HISTÓRIA,354,1MTF,MÉDIA TRIMESTRAL FINAL,RESULTADO DO 1º TRIMESTRE,8.3,2024,0,0,1
3,12,Ensino médio - 3ª Série,Ensino Médio,EDUCAÇÃO REGULAR - ENSINO MÉDIO,22,PIAUÍ,VALE DOS RIOS PIAUÍ E ITAUEIRA,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2202091,CALDEIRAO GRANDE DO PIAUI,...,HISTÓRIA,361,2MTF,MÉDIA TRIMESTRAL FINAL,RESULTADO DO 2º TRIMESTRE,6.7,2024,0,0,1
4,12,Ensino médio - 3ª Série,Ensino Médio,EDUCAÇÃO REGULAR - ENSINO MÉDIO,22,PIAUÍ,VALE DOS RIOS PIAUÍ E ITAUEIRA,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2202091,CALDEIRAO GRANDE DO PIAUI,...,GEOGRAFIA,354,1MTF,MÉDIA TRIMESTRAL FINAL,RESULTADO DO 1º TRIMESTRE,8.8,2024,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3518100,1726,Curso técnico integrado (ensino médio integrad...,Educação Profissional Técnica de Nível Médio -...,ENSINO INTEGRADO INTEGRAL - NOVAS 2024,22,PIAUÍ,ENTRE RIOS,GERENCIA REGIONAL DE EDUCACAO DE TERESINA - 21ª,2211001,TERESINA,...,ATIVIDADES INTEGRADORAS - MONITORIA / HORÁRIO ...,361,2MTF,MÉDIA TRIMESTRAL FINAL,RESULTADO DO 2º TRIMESTRE,6.7,2024,0,0,1
3518101,1726,Curso técnico integrado (ensino médio integrad...,Educação Profissional Técnica de Nível Médio -...,ENSINO INTEGRADO INTEGRAL - NOVAS 2024,22,PIAUÍ,ENTRE RIOS,GERENCIA REGIONAL DE EDUCACAO DE TERESINA - 21ª,2211001,TERESINA,...,ATIVIDADES INTEGRADORAS - CULTURA INTEGRADA A ...,354,1MTF,MÉDIA TRIMESTRAL FINAL,RESULTADO DO 1º TRIMESTRE,7.0,2024,0,0,1
3518102,1726,Curso técnico integrado (ensino médio integrad...,Educação Profissional Técnica de Nível Médio -...,ENSINO INTEGRADO INTEGRAL - NOVAS 2024,22,PIAUÍ,ENTRE RIOS,GERENCIA REGIONAL DE EDUCACAO DE TERESINA - 21ª,2211001,TERESINA,...,ATIVIDADES INTEGRADORAS - CULTURA INTEGRADA A ...,361,2MTF,MÉDIA TRIMESTRAL FINAL,RESULTADO DO 2º TRIMESTRE,6.7,2024,0,0,1
3518103,1726,Curso técnico integrado (ensino médio integrad...,Educação Profissional Técnica de Nível Médio -...,ENSINO INTEGRADO INTEGRAL - NOVAS 2024,22,PIAUÍ,ENTRE RIOS,GERENCIA REGIONAL DE EDUCACAO DE TERESINA - 21ª,2211001,TERESINA,...,ATIVIDADES INTEGRADORAS - INTELIGÊNCIA ARTIFICIAL,354,1MTF,MÉDIA TRIMESTRAL FINAL,RESULTADO DO 1º TRIMESTRE,9.3,2024,0,0,1


8.5 s ± 1.25 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
# Conexão e Dataframe 10.10.1.87
# Query Frequência ITA

import pyodbc
import pandas as pd

%run Caminhos.ipynb

dados_conexao_seduc

try:
    # Abrir conexão
    conexao = pyodbc.connect(dados_conexao_seduc)
    print("Conexão bem sucedida")

    # Executa uma consulta
    cursor = conexao.cursor()

    # Executa o Select
    query = """
    SELECT
        a.idAula,
        a.tipoAula,
        f.tipoFrequencia,
        f.idMatricula,
        t.idTurma,
        m.idEtapa,
        d.nome as 'Componente Curricular',
        pl.anoLetivo as 'Ano'
    FROM Aula a
    LEFT JOIN frequencia f ON f.idAula = a.idAula
    LEFT JOIN matricula m ON m.idMatricula = f.idMatricula
    LEFT JOIN periodoLetivo pl ON pl.idPeriodo = m.idPeriodo
    LEFT JOIN escola es ON es.idEscola = m.idEscola
    LEFT JOIN turmaDisciplina td ON td.idTurmaDisciplina = a.idTurmaDisciplina
    LEFT JOIN disciplina d ON d.idDisciplina = td.idDisciplina
    LEFT JOIN turma t ON t.idTurma = m.idTurma
    WHERE es.dependenciaAdministrativa = 'ESTADUAL'
    AND pl.anoLetivo IN (2024, 2025)
    AND m.situacaoMatricula = 'Cursando'
    AND m.matriculaAtiva = 1
    AND t.idTurma IN (287816, 287818)
    """
    cursor.execute(query)

    # Obter os nomes das colunas
    columns = [desc[0] for desc in cursor.description]
    select = cursor.fetchall()

    # Desmebra a lista de Tuplas em uma tabela
    tabela_frequencia_ITA = pd.DataFrame.from_records(select, columns=columns)

except pyodbc.OperationalError as erro:
    # Captura e trata erros operacionais específicos do pyodbc
    print(f"Erro operacional ao criar a conexão: {erro}")

except Exception as erro:
    # Captura e trata qualquer outro tipo de exceção
    print(f"Erro ao criar a conexão ou executar a consulta: {erro}")

finally:
    # Fecha a conexão com o banco de dados, se estiver aberta
    if cursor:
        cursor.close()
        conexao.close()

%run Caminhos.ipynb

# salvar os dados em um arquivo pickle
#%timeit tabela_frequencia_ITA.to_pickle(f'{pasta_hierarquia}{caminho_tabela_frequencia_pkl}')

# salvar os dados em um arquivo parquet
tabela_frequencia_ITA.to_parquet(f'{pasta_hierarquia}{caminho_tabela_frequencia_ITA_parquet}')

# salvar os dados em um arquivo feather
#%timeit tabela_frequencia_ITA.to_feather(f'{pasta_hierarquia}{caminho_tabela_frequencia_feather}')

#!dir "C:/Users/VeryPC/OneDrive - verytecnologia.com.br/4.PIAUÍ - 1.SEDUC/2. Execução/2.5 Projetos/Escritório de Projetos/4. Painéis/Painel iSEDUC/Indicadores_Hierarquia/Query_indicadores/"

Conexão bem sucedida


In [ ]:
# Conexão e Dataframe 10.10.1.87
# Query Frequência%

import pyodbc
import pandas as pd

%run Caminhos.ipynb

dados_conexao_seduc

try:
    # Abrir conexão
    conexao = pyodbc.connect(dados_conexao_seduc)
    print("Conexão bem sucedida")

    # Executa uma consulta
    cursor = conexao.cursor()

    # Executa o Select
    query = """
    SELECT DISTINCT
        a.idMatricula,
        COUNT(CASE 
            WHEN a.tipoFrequencia = 'Presença' THEN 1 END) AS presencas,
        COUNT(a.tipoFrequencia) AS totalAulas,
        (COUNT(CASE 
            WHEN a.tipoFrequencia = 'Presença' THEN 1 END) * 100.0 / COUNT(a.tipoFrequencia)) AS percPresenca
    FROM
        (SELECT
            a.idAula,
            f.tipoFrequencia,
            f.idMatricula,
            pl.anoLetivo
        FROM Aula a
        LEFT JOIN frequencia f ON f.idAula = a.idAula
        LEFT JOIN matricula m ON m.idMatricula = f.idMatricula
        LEFT JOIN periodoLetivo pl ON pl.idPeriodo = m.idPeriodo
        LEFT JOIN escola es ON es.idEscola = m.idEscola
        LEFT JOIN turmaDisciplina td ON td.idTurmaDisciplina = a.idTurmaDisciplina
        LEFT JOIN disciplina d ON d.idDisciplina = td.idDisciplina
        LEFT JOIN turma t ON t.idTurma = m.idTurma
        WHERE es.dependenciaAdministrativa = 'ESTADUAL'
        AND pl.anoLetivo = 2024
        AND m.situacaoMatricula = 'Cursando'
        AND m.matriculaAtiva = 1)a
        GROUP BY a.idMatricula
    """
    cursor.execute(query)

    # Obter os nomes das colunas
    columns = [desc[0] for desc in cursor.description]
    select = cursor.fetchall()

    # Desmebra a lista de Tuplas em uma tabela
    tabela_frequencia_ativo = pd.DataFrame.from_records(select, columns=columns)

except pyodbc.OperationalError as erro:
    # Captura e trata erros operacionais específicos do pyodbc
    print(f"Erro operacional ao criar a conexão: {erro}")

except Exception as erro:
    # Captura e trata qualquer outro tipo de exceção
    print(f"Erro ao criar a conexão ou executar a consulta: {erro}")

finally:
    # Fecha a conexão com o banco de dados, se estiver aberta
    if cursor:
        cursor.close()
        conexao.close()

%run Caminhos.ipynb

# salvar os dados em um arquivo pickle
#%timeit tabela_frequencia_ativo.to_pickle(f'{pasta_hierarquia}{caminho_tabela_frequencia_pkl}')

# salvar os dados em um arquivo parquet
tabela_frequencia_ativo.to_parquet(f'{pasta_hierarquia}{caminho_tabela_frequencia_ativo_parquet}')

# salvar os dados em um arquivo feather
#%timeit tabela_frequencia_ativo.to_feather(f'{pasta_hierarquia}{caminho_tabela_frequencia_feather}')

#!dir "C:/Users/VeryPC/OneDrive - verytecnologia.com.br/4.PIAUÍ - 1.SEDUC/2. Execução/2.5 Projetos/Escritório de Projetos/4. Painéis/Painel iSEDUC/Indicadores_Hierarquia/Query_indicadores/"

Conexão bem sucedida


In [ ]:
# Conexão e Dataframe 10.10.1.87
# Query Escolas de Tempo Integral

import pyodbc
import pandas as pd

%run Caminhos.ipynb

dados_conexao_seduc

try:
    # Abrir conexão
    conexao = pyodbc.connect(dados_conexao_seduc)
    print("Conexão bem sucedida")

    # Executa uma consulta
    cursor = conexao.cursor()

    # Executa o Select
    query = """
    SELECT DISTINCT
    m.idMatricula,
    est.idEstado as 'Id Estado',
    mu.territorioDesenvolvimento as 'Territorio Desenvolv', 
    gr.nome as 'Gerencia Regional', 
    CAST(mu.idMunicipio as INT) as 'Id Municipio',
    e.idEscola as 'Id Escola', 
    e.inep,
    e.nome,
    e.ativa,
    e.conveniada,
    e.dependenciaAdministrativa,
    e.dataTempoIntegral,
    e.localizacao,
    YEAR(e.dataTempoIntegral) as Ano,
    CASE
        WHEN e.tempoIntegral = 1 AND et.tempoIntegral = 1 THEN 1
        ELSE 0
    END AS STATUS_INTEGRAL,
    et.tempoIntegral
    FROM escola e
    LEFT JOIN turma t ON t.idEscola = e.idEscola
    LEFT JOIN etapa et ON et.idEtapa = t.idEtapa
    LEFT JOIN matricula m ON m.idEscola =  e.idEscola
    LEFT JOIN municipio mu ON mu.idMunicipio = e.idMunicipio 
    LEFT JOIN gerenciaRegional gr ON gr.idGerenciaRegional = e.idGerenciaRegional
    LEFT JOIN estado est ON est.idEstado = mu.idEstado
    """
    cursor.execute(query)

    # Obter os nomes das colunas
    columns = [desc[0] for desc in cursor.description]
    select = cursor.fetchall()

    # Desmebra a lista de Tuplas em uma tabela
    tabela_Tempo_Integral = pd.DataFrame.from_records(select, columns=columns)

except pyodbc.OperationalError as erro:
    # Captura e trata erros operacionais específicos do pyodbc
    print(f"Erro operacional ao criar a conexão: {erro}")

except Exception as erro:
    # Captura e trata qualquer outro tipo de exceção
    print(f"Erro ao criar a conexão ou executar a consulta: {erro}")

finally:
    # Fecha a conexão com o banco de dados, se estiver aberta
    if cursor:
        cursor.close()
        conexao.close()

%run Caminhos.ipynb

# salvar os dados em um arquivo pickle
#%timeit tabela_Tempo_Integral.to_pickle(f'{pasta_hierarquia}{caminho_tabela_frequencia_pkl}')

# salvar os dados em um arquivo parquet
%timeit tabela_Tempo_Integral.to_parquet(f'{pasta_hierarquia}{caminho_tabela_Tempo_Integral_parquet}')

# salvar os dados em um arquivo feather
#%timeit tabela_Tempo_Integral.to_feather(f'{pasta_hierarquia}{caminho_tabela_frequencia_feather}')

#!dir "C:/Users/VeryPC/OneDrive - verytecnologia.com.br/4.PIAUÍ - 1.SEDUC/2. Execução/2.5 Projetos/Escritório de Projetos/4. Painéis/Painel iSEDUC/Indicadores_Hierarquia/Query_indicadores/"

Driver={SQL Server};Server=briskdb.seduc.pi.gov.br,1433;Database=briskdb;UID=usr_briskdb_verytec_bi;PWD=ROcqvkm0Brisk@
Erro ao conectar: ORA-12170: Cannot connect. - timeout of -s for host 10.0.57.18 port 1521. (CONNECTION_ID=tN0jcN5oSV6rgEf0fZANkA==)
Help: https://docs.oracle.com/error-help/db/ora-12170/
Conexão bem sucedida
Driver={SQL Server};Server=briskdb.seduc.pi.gov.br,1433;Database=briskdb;UID=usr_briskdb_verytec_bi;PWD=ROcqvkm0Brisk@
Erro ao conectar: ORA-12170: Cannot connect. - timeout of -s for host 10.0.57.17 port 1521. (CONNECTION_ID=jpezbn1xRj6WB+AMzZJseg==)
Help: https://docs.oracle.com/error-help/db/ora-12170/
1.8 s ± 38 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
# Conexão e Dataframe 10.10.1.87
# Query Lotacao Escola

import pyodbc
import pandas as pd

%run Caminhos.ipynb

dados_conexao_seduc

try:
    # Abrir conexão
    conexao = pyodbc.connect(dados_conexao_seduc)
    print("Conexão bem sucedida")

    # Executa uma consulta
    cursor = conexao.cursor()

    # Executa o Select
    query = """
    SELECT
    pl.anoLetivo as 'Ano',
    e.idEscola,
    e.nome as 'NomeEscola',
    d.idDisciplina,
    d.nome as 'Nome da disciplina',
    l.ativa as 'LotacaoAtiva?',
    COUNT(t.idTurma) AS 'Quantidade de turmas'
    FROM 
        escola e
    LEFT JOIN
        turma t ON t.idEscola = e.idEscola
    LEFT JOIN
        turmaDisciplina td ON td.idTurma = t.idTurma
    LEFT JOIN
        disciplina d ON d.idDisciplina = td.idDisciplina
    LEFT JOIN
        periodoLetivo pl ON t.idPeriodo = pl.idPeriodo
    LEFT JOIN
        lotacao l ON l.idTurmaDisciplina = td.idTurmaDisciplina
    WHERE
        e.dependenciaAdministrativa = 'ESTADUAL'
    AND
        e.ativa = 1
    AND
        e.conveniada = 0
    AND
        t.ativa = 1
    AND
        T.finalizada = 0
    AND
        pl.anoLetivo = 2024
    GROUP BY
        e.idEscola,
        e.nome,
        d.idDisciplina,
        d.nome,
        l.ativa,
        pl.anoLetivo
    ORDER BY
        e.nome,
        d.nome;
    """
    cursor.execute(query)

    # Obter os nomes das colunas
    columns = [desc[0] for desc in cursor.description]
    select = cursor.fetchall()

    # Desmebra a lista de Tuplas em uma tabela
    tabela_lotacao_Turma = pd.DataFrame.from_records(select, columns=columns)

    # Adicionar uma nova coluna com base na condição 2026
    tabela_lotacao_Turma['Matricula_2026'] = tabela_lotacao_Turma.apply(
        lambda row: 1 if row['Ano'] == 2026 else 0,
        axis=1
    )

    # Adicionar uma nova coluna com base na condição 2025
    tabela_lotacao_Turma['Matricula_2025'] = tabela_lotacao_Turma.apply(
        lambda row: 1 if row['Ano'] == 2025 else 0,
        axis=1
    )

    # Adicionar uma nova coluna com base na condição 2024
    tabela_lotacao_Turma['Matricula_2024'] = tabela_lotacao_Turma.apply(
        lambda row: 1 if row['Ano'] == 2024 else 0,
        axis=1
    )

except pyodbc.OperationalError as erro:
    # Captura e trata erros operacionais específicos do pyodbc
    print(f"Erro operacional ao criar a conexão: {erro}")

except Exception as erro:
    # Captura e trata qualquer outro tipo de exceção
    print(f"Erro ao criar a conexão ou executar a consulta: {erro}")

finally:
    # Fecha a conexão com o banco de dados, se estiver aberta
    if cursor:
        cursor.close()
        conexao.close()

display(tabela_lotacao_Turma)
%run Caminhos.ipynb

# salvar os dados em um arquivo pickle
#%timeit tabela_lotacao_Turma.to_pickle(f'{pasta_hierarquia}{caminho_tabela_lotacao_Turma_pkl}')

# salvar os dados em um arquivo parquet
%timeit tabela_lotacao_Turma.to_parquet(f'{pasta_hierarquia}{caminho_tabela_lotacao_Turma_parquet}')

# salvar os dados em um arquivo feather
#%timeit tabela_lotacao_Turma.to_feather(f'{pasta_hierarquia}{caminho_tabela_lotacao_Turma_feather}')

#!dir "C:/Users/VeryPC/OneDrive - verytecnologia.com.br/4.PIAUÍ - 1.SEDUC/2. Execução/2.5 Projetos/Escritório de Projetos/4. Painéis/Painel iSEDUC/Indicadores_Hierarquia/Query_indicadores/"

Conexão bem sucedida


,Ano,idEscola,NomeEscola,idDisciplina,Nome da disciplina,LotacaoAtiva?,Quantidade de turmas,Matricula_2026,Matricula_2025,Matricula_2024
0,2024,279,CEEP CALISTO LOBO,NaN,None,NaN,2,0,0,0
1,2024,279,CEEP CALISTO LOBO,1886.0,ADMINISTRAÇÃO DA PRODUÇÃO,0.0,6,0,0,0
2,2024,279,CEEP CALISTO LOBO,1886.0,ADMINISTRAÇÃO DA PRODUÇÃO,1.0,3,0,0,0
3,2024,279,CEEP CALISTO LOBO,1884.0,ADMINISTRAÇÃO DE MARKETING,1.0,1,0,0,0
4,2024,279,CEEP CALISTO LOBO,2096.0,ADMINISTRAÇÃO DE SISTEMAS OPERACIONAIS,0.0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...
52909,2024,1088,UNIDADE ESCOLAR PROFESSORA MARIA DO SOCORRO DE...,2295.0,TRILHAS DE APRENDIZAGEM INTEGRADA - COMP 1,1.0,1,0,0,0
52910,2024,1088,UNIDADE ESCOLAR PROFESSORA MARIA DO SOCORRO DE...,2296.0,TRILHAS DE APRENDIZAGEM INTEGRADA - COMP 2,1.0,1,0,0,0
52911,2024,1088,UNIDADE ESCOLAR PROFESSORA MARIA DO SOCORRO DE...,2296.0,TRILHAS DE APRENDIZAGEM INTEGRADA - COMP 2,0.0,1,0,0,0
52912,2024,1088,UNIDADE ESCOLAR PROFESSORA MARIA DO SOCORRO DE...,2297.0,TRILHAS DE APRENDIZAGEM INTEGRADA - COMP 3,1.0,1,0,0,0


30.5 ms ± 902 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [7]:
# Conexão e Dataframe 10.10.1.87
# Query Análise banco de dados

import pyodbc
import pandas as pd

%run Caminhos.ipynb

dados_conexao_seduc

try:
    # Abrir conexão
    conexao = pyodbc.connect(dados_conexao_seduc)
    print("Conexão bem sucedida")

    # Executa uma consulta
    cursor = conexao.cursor()

    # Executa o Select
    query = """
    -- Criar uma tabela temporária para armazenar os resultados
    IF OBJECT_ID('tempdb..#TabelaAnalise') IS NOT NULL DROP TABLE #TabelaAnalise;

    CREATE TABLE #TabelaAnalise (
        Nome_Tabela NVARCHAR(255),
        Nome_Coluna NVARCHAR(255),
        Tipo_Dado NVARCHAR(50),
        Qtde_Linhas INT,
        Qtde_Nulos INT,
        Ultima_Atualizacao DATETIME
    );

    DECLARE @sql NVARCHAR(MAX) = '';

    -- Inserir os metadados das colunas na tabela temporária
    SELECT @sql += 
    'INSERT INTO #TabelaAnalise
    SELECT 
        ''' + t.TABLE_SCHEMA + '.' + t.TABLE_NAME + ''' AS Nome_Tabela, 
        c.COLUMN_NAME AS Nome_Coluna,
        c.DATA_TYPE AS Tipo_Dado,
        (SELECT COUNT(*) FROM ' + QUOTENAME(t.TABLE_SCHEMA) + '.' + QUOTENAME(t.TABLE_NAME) + ') AS Qtde_Linhas,
        (SELECT COUNT(*) FROM ' + QUOTENAME(t.TABLE_SCHEMA) + '.' + QUOTENAME(t.TABLE_NAME) + ' WHERE ' + QUOTENAME(c.COLUMN_NAME) + ' IS NULL) AS Qtde_Nulos,
        (SELECT MAX(modify_date) 
            FROM sys.tables st
            INNER JOIN sys.schemas s ON st.schema_id = s.schema_id
            WHERE st.name = ''' + t.TABLE_NAME + ''' AND s.name = ''' + t.TABLE_SCHEMA + ''') AS Ultima_Atualizacao
    FROM INFORMATION_SCHEMA.COLUMNS c
    WHERE c.TABLE_NAME = ''' + t.TABLE_NAME + ''' AND c.TABLE_SCHEMA = ''' + t.TABLE_SCHEMA + '''; '
    FROM INFORMATION_SCHEMA.TABLES t
    JOIN INFORMATION_SCHEMA.COLUMNS c 
        ON t.TABLE_NAME = c.TABLE_NAME AND t.TABLE_SCHEMA = c.TABLE_SCHEMA
    WHERE t.TABLE_TYPE = 'BASE TABLE';

    -- Executar a query dinâmica
    EXEC sp_executesql @sql;

    -- Retornar os resultados
    SELECT * FROM #TabelaAnalise;

    -- Limpar a tabela temporária
    DROP TABLE #TabelaAnalise;
    """
    cursor.execute(query)

    # Obter os nomes das colunas
    columns = [desc[0] for desc in cursor.description]
    select = cursor.fetchall()

    # Desmebra a lista de Tuplas em uma tabela
    tabela_banco_dados = pd.DataFrame.from_records(select, columns=columns)

except pyodbc.OperationalError as erro:
    # Captura e trata erros operacionais específicos do pyodbc
    print(f"Erro operacional ao criar a conexão: {erro}")

except Exception as erro:
    # Captura e trata qualquer outro tipo de exceção
    print(f"Erro ao criar a conexão ou executar a consulta: {erro}")

finally:
    # Fecha a conexão com o banco de dados, se estiver aberta
    if cursor:
        cursor.close()
        conexao.close()

display(tabela_banco_dados)
%run Caminhos.ipynb

# salvar os dados em um arquivo parquet
%timeit tabela_banco_dados.to_parquet(f'{pasta_hierarquia}{caminho_tabela_banco_dados_parquet}')

Driver={SQL Server};Server=briskdb.seduc.pi.gov.br,1433;Database=briskdb;UID=usr_briskdb_verytec_bi;PWD=ROcqvkm0Brisk@
Erro ao conectar: ORA-12170: Cannot connect. - timeout of -s for host 10.0.57.16 port 1521. (CONNECTION_ID=jAeF70jmRHmMsxHcBVXpEQ==)
Help: https://docs.oracle.com/error-help/db/ora-12170/
Conexão bem sucedida
Erro ao criar a conexão ou executar a consulta: 'NoneType' object is not iterable


None

Erro ao conectar: ORA-12545: Connect failed because target host or object does not exist
Help: https://docs.oracle.com/error-help/db/ora-12545/


AttributeError: 'NoneType' object has no attribute 'to_parquet'